# Latent-OOD outlier characterization (TID-based, corrected)

**Discipline:** catalogue key is **TARGETID**, never a positional index. `get_outliers` emits `outlier_target_ids`; we match the FastSpecFit VAC by TARGETID. (The earlier numeric `chunk*1024+row` decode was a bug — the spender loader is `sorted(glob)` lexicographic.)

Detector: IsolationForest on Cue-mock latents (S/N>3), scoring DESI. Outlier set: `desi_outliers_cue_snr3.pt`.

In [ ]:
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from astropy.table import Table
from hubersed.paths import PATHS

DATA, RES = PATHS["DATA"], PATHS["RESULTS"]
VAC = DATA / "fastspec-iron-sv3-bright.fits"
CUE = DATA / "prospector_model" / "prospector_stochastic_model_seds_cue_500000.h5"
SNR = 3.0
LINES = ["HBETA", "OIII_5007", "HALPHA", "NII_6584", "SII_6716", "SII_6731"]
print("VAC exists:", VAC.exists())

In [ ]:
# --- outliers + parent, by TARGETID ---
blob = torch.load(RES / "desi_outliers_cue_snr3.pt", weights_only=False)
out_tid = np.asarray(blob["outlier_target_ids"]).astype(np.int64)
par_tid = np.asarray(blob["desi_target_ids"]).astype(
    np.int64
)  # parent = S/N>3 DESI pool
print("outliers:", out_tid.size, "  parent (S/N>3):", par_tid.size)

In [ ]:
# --- VAC match by TARGETID; grab line fluxes + SNR (=flux*sqrt(ivar)) + stellar params ---
cols = (
    ["TARGETID", "HALPHA_EW", "DN4000", "LOGMSTAR", "SFR"]
    + [f"{l}_FLUX" for l in LINES]
    + [f"{l}_FLUX_IVAR" for l in LINES]
)
vac = Table.read(VAC, hdu="FASTSPEC")[cols]
vtid = np.asarray(vac["TARGETID"], np.int64)
order = np.argsort(vtid)
vts = vtid[order]


def rows_for(t):
    pos = np.clip(np.searchsorted(vts, t), 0, len(vts) - 1)
    return order[pos], vts[pos] == t


def grab(t):
    r, ok = rows_for(t)
    d = {}
    for l in LINES:
        f = np.asarray(vac[f"{l}_FLUX"])[r]
        iv = np.asarray(vac[f"{l}_FLUX_IVAR"])[r]
        d[l] = np.where(ok, f, np.nan)
        d[l + "_SNR"] = np.where(ok, f * np.sqrt(np.clip(iv, 0, None)), 0.0)
    for c in ["HALPHA_EW", "DN4000", "LOGMSTAR", "SFR"]:
        d[c] = np.where(ok, np.asarray(vac[c])[r], np.nan)
    return d, ok


d_out, ok_out = grab(out_tid)
d_all, ok_all = grab(par_tid)
print(
    f"matched: outliers {ok_out.sum()}/{out_tid.size}   parent {ok_all.sum()}/{par_tid.size}"
)

## BPT (NII, SII) + classification

In [ ]:
def kauff03(x):
    return 0.61 / (x - 0.05) + 1.30


def kewl01n(x):
    return 0.61 / (x - 0.47) + 1.19


def kewl01s(x):
    return 0.72 / (x - 0.32) + 1.30


def kewl06(x):
    return 1.89 * x + 0.76


def bpt_coords(d, kind):
    with np.errstate(all="ignore"):
        y = np.log10(d["OIII_5007"] / d["HBETA"])
        if kind == "NII":
            x = np.log10(d["NII_6584"] / d["HALPHA"])
            sel = (d["NII_6584_SNR"] >= SNR) & (d["HALPHA_SNR"] >= SNR)
        else:
            x = np.log10((d["SII_6716"] + d["SII_6731"]) / d["HALPHA"])
            sel = (
                (d["SII_6716_SNR"] >= SNR)
                & (d["SII_6731_SNR"] >= SNR)
                & (d["HALPHA_SNR"] >= SNR)
            )
        sel = (
            sel
            & (d["OIII_5007_SNR"] >= SNR)
            & (d["HBETA_SNR"] >= SNR)
            & np.isfinite(x)
            & np.isfinite(y)
        )
    return x, y, sel


fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, kind, curves in [
    (
        ax[0],
        "NII",
        [
            (kauff03, -1.8, 0.0, "k--", "Kauffmann+03"),
            (kewl01n, -1.8, 0.35, "b-", "Kewley+01"),
        ],
    ),
    (
        ax[1],
        "SII",
        [
            (kewl01s, -1.8, 0.1, "b-", "Kewley+01"),
            (kewl06, -0.3, 0.8, "g-.", "Kewley+06 Sy/LINER"),
        ],
    ),
]:
    xa, ya, sa = bpt_coords(d_all, kind)
    xo, yo, so = bpt_coords(d_out, kind)
    a.hexbin(xa[sa], ya[sa], gridsize=80, bins="log", cmap="Greys", mincnt=1)
    a.scatter(
        xo[so], yo[so], s=8, c="red", lw=0, label=f"outliers (n={so.sum()})", zorder=5
    )
    for fn, lo, hi, st, lb in curves:
        xx = np.linspace(lo, hi, 200)
        a.plot(xx, fn(xx), st, lw=1.4, label=lb)
    a.set_xlim(-1.8, 0.8)
    a.set_ylim(-1.2, 1.5)
    a.set_xlabel("log [NII]/Ha" if kind == "NII" else "log [SII]/Ha")
    a.set_ylabel("log [OIII]/Hb")
    a.set_title(f"{kind}-BPT")
    a.legend(loc="lower left", fontsize=8)
fig.tight_layout()
fig.savefig(RES / "outlier_bpt.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# NII-BPT classification (low-[NII] points above Kewley = low-Z SF, NOT AGN)
def classify(d, ntot, name):
    x, y, sel = bpt_coords(d, "NII")
    xs, ys = x[sel], y[sel]
    sf = (ys < kauff03(xs)) & (xs < 0.05)
    agn = (ys > kewl01n(xs)) | (xs > 0.47)
    comp = ~sf & ~agn
    print(
        f"{name:8} 4-line {sel.sum()}/{ntot} ({100 * sel.sum() / ntot:.0f}%)  SF {100 * sf.mean():.1f}%  comp {100 * comp.mean():.1f}%  AGN {100 * agn.mean():.1f}%"
    )
    if agn.sum():
        print(
            f'         {agn.sum()} "AGN" median log[NII]/Ha={np.median(xs[agn]):.2f} (real AGN >0; low => low-Z SF)'
        )


classify(d_out, out_tid.size, "outliers")
classify(d_all, par_tid.size, "parent")

## Halpha EW enrichment

In [ ]:
eo = d_out["HALPHA_EW"][ok_out]
ea = d_all["HALPHA_EW"][ok_all]
eo = eo[eo > 0]
ea = ea[ea > 0]
print(f"median Ha EW: outliers {np.median(eo):.0f} A   parent {np.median(ea):.0f} A")
for thr in [50, 100, 200, 300]:
    fo = np.mean(eo > thr)
    fa = np.mean(ea > thr)
    print(
        f"  EW>{thr:4d}: outliers {100 * fo:5.1f}%   parent {100 * fa:5.2f}%   enrich {fo / max(fa, 1e-9):.0f}x"
    )
fig, ax = plt.subplots(figsize=(7, 4))
b = np.logspace(-1, np.log10(max(eo.max(), ea.max())), 60)
ax.hist(
    ea, bins=b, density=True, histtype="step", color="k", label=f"parent (n={ea.size})"
)
ax.hist(
    eo,
    bins=b,
    density=True,
    histtype="stepfilled",
    color="red",
    alpha=0.4,
    label=f"outliers (n={eo.size})",
)
ax.set_xscale("log")
ax.set_xlabel("Ha EW [A]")
ax.set_ylabel("density")
ax.legend()
fig.savefig(RES / "outlier_ew.pdf", bbox_inches="tight")
plt.show()

## Mock (Cue) prior coverage: gap vs misspecification vs density imbalance

In [ ]:
with h5py.File(CUE, "r") as hf:
    sm = hf["priors/stellar_masses"][:]
    gz = hf["priors/gas_metallicities"][:]
    gu = hf["priors/gas_ionization_parameters"][:]
sm_log = sm if np.nanmedian(sm) < 20 else np.log10(sm)
p = lambda a: np.round(np.nanpercentile(a, [0.5, 50, 99.5]), 2)
print(
    "mock log M*    p0.5/50/99.5 =",
    p(sm_log),
    " min",
    round(float(np.nanmin(sm_log)), 2),
)
print("outlier logM*  p0.5/50/99.5 =", p(d_out["LOGMSTAR"][ok_out]))
print(
    f"mock logM*<9: {100 * np.mean(sm_log < 9):.1f}%   <8.5: {100 * np.mean(sm_log < 8.5):.1f}%"
)
print("mock gas_logZ  p0.5/50/99.5 =", p(gz), "(floor -2.2)")
print("mock gas_logU  p0.5/50/99.5 =", p(gu), "(ceiling -1)")
print("=> marginals cover the outliers; check the EW density next.")

In [ ]:
# Mock Halpha EW from Cue spectra (contiguous read) vs DESI
with h5py.File(CUE, "r") as hf:
    wv = hf["wavelength"][:]
    zall = hf["priors/redshifts"][:]
    fx = hf["fluxes"][:40000]
    zs = zall[:40000]
keep = zs < 0.35
fx = fx[keep]
zs = zs[keep]


def mock_ha_ew(fl, zz):
    xr = wv / (1 + zz)
    cm = ((xr > 6500) & (xr < 6540)) | ((xr > 6595) & (xr < 6630))
    lm = (xr > 6556) & (xr < 6573)
    if cm.sum() < 5 or lm.sum() < 3:
        return np.nan
    c = np.median(fl[cm].astype(np.float64))
    if c <= 0:
        return np.nan
    dl = np.gradient(xr)
    return float(np.sum((fl[lm].astype(np.float64) - c) / c * dl[lm]))


ew_mock = np.array([mock_ha_ew(fx[i], zs[i]) for i in range(len(zs))])
ew_mock = ew_mock[np.isfinite(ew_mock) & (ew_mock > 0)]
print("Ha EW  p50/90/99/max:")
print(
    f"  mock     {np.percentile(ew_mock, [50, 90, 99]).round(0)}  {ew_mock.max():.0f}  (n={ew_mock.size})"
)
print(f"  outliers {np.percentile(eo, [50, 90, 99]).round(0)}  {eo.max():.0f}")
for thr in [100, 200, 300]:
    print(
        f"  EW>{thr}: mock {100 * np.mean(ew_mock > thr):.2f}%   outliers {100 * np.mean(eo > thr):.0f}%"
    )
print(
    "=> model CAN make extreme EW (mock max ~1200) but prior UNDER-SAMPLES it => density imbalance in SFH/sSFR prior."
)

## FSPS (Byler) vs Cue outlier set

In [ ]:
fsps_f = RES / "desi_outliers_fsps_snr3.pt"
if fsps_f.exists():
    ft = set(
        int(x) for x in torch.load(fsps_f, weights_only=False)["outlier_target_ids"]
    )
    ct = set(int(x) for x in out_tid)
    inter = ft & ct
    print(
        f"FSPS {len(ft)} | Cue {len(ct)} | overlap {len(inter)} ({100 * len(inter) / min(len(ft), len(ct)):.0f}% of smaller)"
    )
    df, okf = grab(np.array(sorted(ft), dtype=np.int64))
    ewf = df["HALPHA_EW"][okf]
    ewf = ewf[ewf > 0]
    print(
        f"FSPS outliers: median Ha EW {np.median(ewf):.0f} A, EW>100 {100 * np.mean(ewf > 100):.0f}%, median logM* {np.nanmedian(df['LOGMSTAR'][okf]):.2f}"
    )
else:
    print(
        "desi_outliers_fsps_snr3.pt not found yet -- encode FSPS mocks + run get_outliers first."
    )